# Vector Search Setup for RAG Pipeline

**Purpose:** Create embeddings and Vector Search index for natural language querying

**Process:**
1. Build document corpus from facilities
2. Create Delta table with documents
3. Create Vector Search endpoint  
4. Create Vector Search index with embeddings
5. Test similarity search

**Output:** 
* Table: `virtue_foundation.ghana.facility_documents`
* Index: `virtue_foundation.ghana.facility_embeddings`

In [0]:
from pyspark.sql import functions as F
from databricks.vector_search.client import VectorSearchClient

CATALOG = "virtue_foundation"
SCHEMA = "ghana"
SOURCE_TABLE = "facilities_silver"
DOCUMENT_TABLE = "facility_documents"
VECTOR_SEARCH_ENDPOINT = "facility_search_endpoint"
INDEX_NAME = "facility_embeddings"

print(f"Source: {CATALOG}.{SCHEMA}.{SOURCE_TABLE}")
print(f"Document table: {CATALOG}.{SCHEMA}.{DOCUMENT_TABLE}")

In [0]:
df_silver = spark.table(f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}")
print(f"✅ Loaded {df_silver.count():,} facilities")

In [0]:
# Helper to format arrays
def format_array(arr):
    if arr is None or len(arr) == 0:
        return "None listed"
    return ", ".join(arr)

format_array_udf = F.udf(format_array)

# Build document text
df_documents = df_silver.withColumn(
    "document_text",
    F.concat_ws(
        "\n",
        F.concat(F.lit("Facility: "), F.coalesce(F.col("name"), F.lit("Unnamed"))),
        F.concat(F.lit("Type: "), F.coalesce(F.col("facilityTypeId"), F.lit("Unknown"))),
        F.concat(F.lit("Location: "), 
                 F.coalesce(F.col("address_city"), F.lit("")), F.lit(", "),
                 F.coalesce(F.col("address_stateOrRegion"), F.lit("")), F.lit(", Ghana")),
        F.concat(F.lit("Specialties: "),
                 F.when(F.col("specialties").isNotNull(), format_array_udf(F.col("specialties")))
                  .otherwise(F.lit("None"))),
        F.concat(F.lit("Procedures: "),
                 F.when(F.col("procedure").isNotNull(), format_array_udf(F.col("procedure")))
                  .otherwise(F.lit("None"))),
        F.concat(F.lit("Equipment: "),
                 F.when(F.col("equipment").isNotNull(), format_array_udf(F.col("equipment")))
                  .otherwise(F.lit("None"))),
        F.concat(F.lit("Doctors: "), F.coalesce(F.col("numberDoctors").cast("string"), F.lit("N/A")),
                 F.lit(" | Beds: "), F.coalesce(F.col("capacity").cast("string"), F.lit("N/A"))),
        F.when(F.col("description").isNotNull(),
               F.concat(F.lit("Description: "), F.col("description"))).otherwise(F.lit(""))
    )
)

print("✅ Document text created")

In [0]:
# Select columns for Vector Search
df_doc_table = df_documents.select(
    F.col("unique_id").alias("id"),
    "document_text",
    "name", "facilityTypeId", "address_city", "address_stateOrRegion",
    "numberDoctors", "capacity", "specialties",
    F.current_timestamp().alias("indexed_at")
)

print(f"✅ Document table prepared: {df_doc_table.count():,} documents")

In [0]:
full_table_name = f"{CATALOG}.{SCHEMA}.{DOCUMENT_TABLE}"

df_doc_table.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(full_table_name)

print(f"✅ Document table created: {full_table_name}")
print("   Change Data Feed: ENABLED")

In [0]:
%sql
COMMENT ON TABLE virtue_foundation.ghana.facility_documents IS
'Document corpus for Vector Search RAG pipeline. Embeddings via databricks-gte-large-en.';

ALTER TABLE virtue_foundation.ghana.facility_documents
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [0]:
vsc = VectorSearchClient()
print("✅ Vector Search client initialized")

In [0]:
# Create or get endpoint
try:
    endpoints = vsc.list_endpoints()
    endpoint_names = [ep['name'] for ep in endpoints.get('endpoints', [])]
    
    if VECTOR_SEARCH_ENDPOINT in endpoint_names:
        print(f"ℹ️  Endpoint '{VECTOR_SEARCH_ENDPOINT}' exists")
    else:
        endpoint = vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT, endpoint_type="STANDARD")
        print(f"✅ Endpoint created: {VECTOR_SEARCH_ENDPOINT}")
    
    vsc.wait_endpoint_ready(VECTOR_SEARCH_ENDPOINT)
    print("✅ Endpoint ready")
except Exception as e:
    print(f"⚠️  Endpoint note: {e}")

In [0]:
full_index_name = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

try:
    # Check if exists
    try:
        vsc.get_index(full_index_name)
        print(f"ℹ️  Index exists, syncing...")
        vsc.sync_index(full_index_name)
        print("✅ Index synced")
    except:
        # Create new index
        print(f"Creating index: {full_index_name}")
        index = vsc.create_delta_sync_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=full_index_name,
            source_table_name=full_table_name,
            pipeline_type="TRIGGERED",
            primary_key="id",
            embedding_source_column="document_text",
            embedding_model_endpoint_name="databricks-gte-large-en"
        )
        print(f"✅ Index created: {full_index_name}")
except Exception as e:
    print(f"❌ Error: {e}")
    raise

In [0]:
# Test similarity search
test_queries = [
    "hospitals with surgery",
    "pediatric care in Accra",
    "emergency services"
]

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    try:
        results = vsc.similarity_search(
            index_name=full_index_name,
            query_text=query,
            columns=["name", "facilityTypeId", "address_city"],
            num_results=3
        )
        if results and 'result' in results:
            for i, r in enumerate(results['result']['data_array'], 1):
                print(f"  {i}. {r.get('name')} ({r.get('facilityTypeId')}) - {r.get('address_city')}")
    except Exception as e:
        print(f"  Error: {e}")

## ✅ Vector Search Complete!

**Created:**
* Document table: `virtue_foundation.ghana.facility_documents`
* Vector index: `virtue_foundation.ghana.facility_embeddings`
* Endpoint: `facility_search_endpoint`
* Model: `databricks-gte-large-en`

**Next:** Build RAG agent using this index

# Vector Search Setup for RAG Pipeline

**Purpose:** Create embeddings and Vector Search index for natural language querying of facilities

**Process:**
1. Build document corpus from facilities (name + specialties + procedures + equipment + description)
2. Create Delta table with documents and metadata
3. Create Vector Search endpoint
4. Create Vector Search index with embeddings
5. Test similarity search

**Input:** `virtue_foundation.ghana.facilities_silver`

**Output:** 
* Delta table: `virtue_foundation.ghana.facility_documents`
* Vector Search index: `facility_embeddings`

**Embedding Model:** `databricks-gte-large-en` (Databricks Foundation Model)

## 1. Setup and Configuration

In [0]:
from pyspark.sql import functions as F
from databricks.vector_search.client import VectorSearchClient
import json

# Configuration
CATALOG = "virtue_foundation"
SCHEMA = "ghana"
SOURCE_TABLE = "facilities_silver"
DOCUMENT_TABLE = "facility_documents"
VECTOR_SEARCH_ENDPOINT = "facility_search_endpoint"
INDEX_NAME = "facility_embeddings"

print(f"Source: {CATALOG}.{SCHEMA}.{SOURCE_TABLE}")
print(f"Document table: {CATALOG}.{SCHEMA}.{DOCUMENT_TABLE}")
print(f"Index: {CATALOG}.{SCHEMA}.{INDEX_NAME}")

## 2. Build Document Corpus

Create rich text documents for each facility combining:
* Name and type
* Location information
* Specialties
* Procedures
* Equipment
* Capabilities
* Description

In [0]:
# Read Silver table
df_silver = spark.table(f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}")

print(f"✅ Loaded {df_silver.count():,} facilities from Silver table")

In [0]:
# Helper function to format array as text
def format_array(arr):
    """Convert array to comma-separated string, handling nulls."""
    if arr is None or len(arr) == 0:
        return "None listed"
    return ", ".join(arr)

format_array_udf = F.udf(format_array)

# Build comprehensive document text for each facility
df_documents = df_silver.withColumn(
    "document_text",
    F.concat_ws(
        "\n",
        # Header
        F.concat(F.lit("Facility: "), F.coalesce(F.col("name"), F.lit("Unnamed Facility"))),
        F.concat(F.lit("Type: "), F.coalesce(F.col("facilityTypeId"), F.lit("Unknown"))),
        F.concat(F.lit("Operator: "), F.coalesce(F.col("operatorTypeId"), F.lit("Unknown"))),
        
        # Location
        F.concat(
            F.lit("Location: "),
            F.coalesce(F.col("address_city"), F.lit("")),
            F.lit(", "),
            F.coalesce(F.col("address_stateOrRegion"), F.lit("")),
            F.lit(", Ghana")
        ),
        
        # Specialties
        F.concat(
            F.lit("Specialties: "),
            F.when(
                F.col("specialties").isNotNull(),
                format_array_udf(F.col("specialties"))
            ).otherwise(F.lit("None listed"))
        ),
        
        # Procedures
        F.concat(
            F.lit("Procedures: "),
            F.when(
                F.col("procedure").isNotNull(),
                format_array_udf(F.col("procedure"))
            ).otherwise(F.lit("None listed"))
        ),
        
        # Equipment
        F.concat(
            F.lit("Equipment: "),
            F.when(
                F.col("equipment").isNotNull(),
                format_array_udf(F.col("equipment"))
            ).otherwise(F.lit("None listed"))
        ),
        
        # Capabilities
        F.concat(
            F.lit("Capabilities: "),
            F.when(
                F.col("capability").isNotNull(),
                format_array_udf(F.col("capability"))
            ).otherwise(F.lit("None listed"))
        ),
        
        # Capacity info
        F.concat(
            F.lit("Doctors: "),
            F.coalesce(F.col("numberDoctors").cast("string"), F.lit("Not reported")),
            F.lit(" | Beds: "),
            F.coalesce(F.col("capacity").cast("string"), F.lit("Not reported"))
        ),
        
        # Description
        F.when(
            F.col("description").isNotNull(),
            F.concat(F.lit("Description: "), F.col("description"))
        ).otherwise(F.lit(""))
    )
)

print("✅ Document text created for all facilities")

# Show sample
print("\nSample document:")
sample_doc = df_documents.select("name", "document_text").first()
print("="*80)
print(sample_doc["document_text"])
print("="*80)

## 3. Prepare Metadata for Vector Search

Select key columns to store as metadata for retrieval and filtering.

In [0]:
# Select columns for document table
# Vector Search will use document_text for embeddings
# Metadata columns are returned with search results
df_doc_table = df_documents.select(
    F.col("unique_id").alias("id"),  # Primary key
    "document_text",  # Text to embed
    "name",
    "facilityTypeId",
    "operatorTypeId",
    "address_city",
    "address_stateOrRegion",
    "address_country",
    "numberDoctors",
    "capacity",
    "has_procedures",
    "has_equipment",
    "has_capability",
    "completeness_score",
    "specialties",
    "procedure",
    "equipment",
    F.current_timestamp().alias("indexed_at")
)

print(f"✅ Document table prepared")
print(f"Total documents: {df_doc_table.count():,}")
print(f"Columns: {len(df_doc_table.columns)}")

## 4. Write Document Table to Delta

Create Delta table for Vector Search to read from.

In [0]:
# Write document table
full_table_name = f"{CATALOG}.{SCHEMA}.{DOCUMENT_TABLE}"

try:
    df_doc_table.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable(full_table_name)
    
    print(f"✅ Document table created: {full_table_name}")
    print(f"   Change Data Feed: ENABLED (required for Vector Search)")
    
except Exception as e:
    print(f"❌ Error writing document table: {e}")
    raise

In [0]:
%sql
-- Add comment to document table
COMMENT ON TABLE virtue_foundation.ghana.facility_documents IS
'Document corpus for Vector Search RAG pipeline.
Each row contains a facility with combined text for semantic search.
Embeddings generated using databricks-gte-large-en model.';

ALTER TABLE virtue_foundation.ghana.facility_documents
SET TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'quality' = 'vector_search_ready'
);

## 5. Create Vector Search Endpoint

Create or retrieve endpoint for hosting the index.

In [0]:
# Initialize Vector Search client
try:
    vsc = VectorSearchClient()
    print("✅ Vector Search client initialized")
except Exception as e:
    print(f"❌ Error initializing Vector Search client: {e}")
    print("Ensure you have appropriate permissions and Vector Search is enabled.")
    raise

In [0]:
# Create Vector Search endpoint if it doesn't exist
try:
    # List existing endpoints
    endpoints = vsc.list_endpoints()
    endpoint_names = [ep['name'] for ep in endpoints.get('endpoints', [])]
    
    if VECTOR_SEARCH_ENDPOINT in endpoint_names:
        print(f"ℹ️  Endpoint '{VECTOR_SEARCH_ENDPOINT}' already exists")
        endpoint = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT)
    else:
        print(f"Creating new endpoint: {VECTOR_SEARCH_ENDPOINT}")
        endpoint = vsc.create_endpoint(
            name=VECTOR_SEARCH_ENDPOINT,
            endpoint_type="STANDARD"
        )
        print(f"✅ Endpoint created: {VECTOR_SEARCH_ENDPOINT}")
    
    # Wait for endpoint to be ready
    print("Waiting for endpoint to be ready...")
    vsc.wait_endpoint_ready(VECTOR_SEARCH_ENDPOINT)
    print("✅ Endpoint is ready")
    
except Exception as e:
    print(f"⚠️  Endpoint creation note: {e}")
    print("If endpoint already exists or you don't have permissions, that's okay.")
    print("You may need workspace admin privileges to create endpoints.")

## 6. Create Vector Search Index

Create the index with embeddings using Databricks Foundation Model.

In [0]:
# Create Vector Search index
full_index_name = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

try:
    # Check if index exists
    try:
        existing_index = vsc.get_index(full_index_name)
        print(f"ℹ️  Index '{full_index_name}' already exists")
        print("Syncing index with updated data...")
        vsc.sync_index(full_index_name)
        print("✅ Index synced")
    except:
        # Index doesn't exist, create it
        print(f"Creating Vector Search index: {full_index_name}")
        print(f"Source table: {full_table_name}")
        print(f"Embedding model: databricks-gte-large-en")
        
        index = vsc.create_delta_sync_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=full_index_name,
            source_table_name=full_table_name,
            pipeline_type="TRIGGERED",
            primary_key="id",
            embedding_source_column="document_text",
            embedding_model_endpoint_name="databricks-gte-large-en"
        )
        
        print(f"✅ Index created: {full_index_name}")
        print("Waiting for index to sync (this may take a few minutes)...")
        
        # Wait for initial sync
        import time
        max_wait = 300  # 5 minutes
        start_time = time.time()
        
        while time.time() - start_time < max_wait:
            index_status = vsc.get_index(full_index_name)
            if index_status.get('status', {}).get('indexed_row_count', 0) > 0:
                print(f"✅ Index synced! Indexed {index_status['status']['indexed_row_count']} rows")
                break
            print(f"Indexing in progress... (waited {int(time.time() - start_time)}s)")
            time.sleep(10)
        
except Exception as e:
    print(f"❌ Error creating/syncing index: {e}")
    print("\nTroubleshooting:")
    print("1. Ensure Vector Search is enabled in your workspace")
    print("2. Verify endpoint exists and is ready")
    print("3. Check that Change Data Feed is enabled on source table")
    print("4. Verify you have CREATE INDEX permissions")
    raise

## 7. Test Vector Search

Perform test searches to validate the index.

In [0]:
# Test similarity search
print("\nTEST QUERIES:")
print("="*80)

test_queries = [
    "hospitals with surgery capabilities",
    "facilities offering pediatric care in Accra",
    "clinics with emergency services"
]

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 80)
    
    try:
        results = vsc.similarity_search(
            index_name=full_index_name,
            query_text=query,
            columns=["name", "facilityTypeId", "address_city", "address_stateOrRegion"],
            num_results=3
        )
        
        if results and 'result' in results and 'data_array' in results['result']:
            for i, result in enumerate(results['result']['data_array'], 1):
                print(f"  {i}. {result.get('name', 'Unknown')} ({result.get('facilityTypeId', 'N/A')})")
                print(f"     Location: {result.get('address_city', 'N/A')}, {result.get('address_stateOrRegion', 'N/A')}")
                print(f"     Score: {result.get('score', 'N/A'):.4f}")
        else:
            print("  No results found")
            
    except Exception as e:
        print(f"  Error: {e}")

print("\n" + "="*80)